# Week 10, Day 4: Linear filters and the linear-nonlinear-Poisson model

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w010/d4.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Simulate an LNP neuron

Write `generator(stim, filt)`, the causal convolution (the first T samples of `np.convolve`). Then compute the rate with an exponential nonlinearity, `0.1 * exp(1.2 g)` spikes per bin, and draw Poisson spike counts in `spikes`. Plot a stretch of stimulus, generator, rate and spikes.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2001)
T, L = 20000, 20                         # 20,000 bins of 10 ms; filter spans 200 ms
lag = np.arange(L)
filt = np.exp(-(lag - 4) ** 2 / 4) - 0.5 * np.exp(-(lag - 9) ** 2 / 9)    # biphasic
filt /= np.linalg.norm(filt)
stim = rng.normal(size=T)                # Gaussian white noise (flicker intensity)

def generator(stim, filt):
    return ...

g = generator(stim, filt)
rate = ...
spikes = ...

seg = slice(1000, 1150)
fig, ax = plt.subplots(3, 1, figsize=(7, 4.5), sharex=True)
ax[0].plot(stim[seg], color="0.5"); ax[0].set_ylabel("stimulus")
ax[1].plot(g[seg]); ax[1].set_ylabel("generator")
ax[2].plot(rate[seg], color="C1"); ax[2].vlines(np.nonzero(spikes[seg])[0], 0, -0.3, color="C3"); ax[2].set_ylabel("rate, spikes")
ax[2].set_xlabel("bin (10 ms)"); plt.tight_layout(); plt.show()
print("total spikes", spikes.sum(), " mean rate", round(rate.mean() / 0.01, 1), "Hz")

In [ ]:
import numpy as np
def test_drive():
    "generator matches the causal convolution sum at a few times"
    g = generator(stim, filt)
    for tt in [30, 500, 4321]:
        assert np.isclose(g[tt], sum(filt[k] * stim[tt - k] for k in range(L)))

def test_counts():
    "spikes is a Poisson draw at the model's rate: its mean is close to the mean rate"
    assert spikes.shape == (T,)
    assert abs(spikes.mean() - rate.mean()) / rate.mean() < 0.05

test_drive()
test_counts()
print("All checks passed")

### Exercise: The spike-triggered average

Compute the STA for lags 0 to L − 1, skipping the first L bins so every window is complete. Compare its shape with the true filter (cosine similarity, and a plot of both normalised to unit length).

In [ ]:
sta = np.zeros(L)
for k in range(L):
    sta[k] = ...

cos = sta @ filt / (np.linalg.norm(sta) * np.linalg.norm(filt))
plt.figure(figsize=(5, 3)); plt.plot(lag * 10, filt, "o-", label="true filter"); plt.plot(lag * 10, sta / np.linalg.norm(sta), "s--", label="STA (normalised)")
plt.xlabel("time before spike (ms)"); plt.legend(frameon=False); plt.show()
print("cosine similarity:", round(cos, 4))

In [ ]:
import numpy as np
def test_sta():
    "sta has one value per lag and matches the definition"
    assert sta.shape == (L,)
    k = 3
    assert np.isclose(sta[k], np.sum(spikes[L:] * stim[L - k:T - k]) / spikes[L:].sum())

def test_shape():
    "the STA points along the true filter: cosine similarity above 0.98"
    assert sta @ filt / (np.linalg.norm(sta) * np.linalg.norm(filt)) > 0.98

test_sta()
test_shape()
print("All checks passed")

### Exercise: Measure the nonlinearity, then predict new data

Using only the first 80% of bins (training), compute the generator with the unit-length STA, split its values into 15 bins with equal numbers of samples (quantile edges), and store the mean spike count in each bin in `nl_hat`. Then predict the rate in every held-out bin from its generator value's bin (`pred`), and compare with the true rate.

In [ ]:
train_end = int(0.8 * T)
sta_unit = sta / np.linalg.norm(sta)
g_hat = generator(stim, sta_unit)
edges = np.quantile(g_hat[:train_end], np.linspace(0, 1, 16))
which = np.clip(np.digitize(g_hat, edges) - 1, 0, 14)          # bin index of every time step
nl_hat = ...
centres = np.array([g_hat[:train_end][which[:train_end] == b].mean() for b in range(15)])
pred = ...

fig, ax = plt.subplots(1, 2, figsize=(9, 3))
ax[0].plot(centres, nl_hat, "o", label="measured"); ax[0].plot(centres, 0.1 * np.exp(1.2 * centres * (sta_unit @ filt)), label="true (rescaled)")
ax[0].set_xlabel("generator signal"); ax[0].set_ylabel("spikes per bin"); ax[0].legend(frameon=False)
ax[1].plot(rate[train_end:train_end + 200], label="true rate"); ax[1].plot(pred[:200], label="predicted"); ax[1].set_xlabel("held-out bin"); ax[1].legend(frameon=False)
plt.tight_layout(); plt.show()
print("held-out correlation with the true rate:", np.corrcoef(pred, rate[train_end:])[0, 1].round(3))

In [ ]:
import numpy as np
def test_bins():
    "nl_hat has one mean spike count per bin, rising with the generator signal"
    assert nl_hat.shape == (15,)
    assert nl_hat[-1] > 5 * nl_hat[0]

def test_predict():
    "predicted rates on the held-out 20% correlate with the true rate above 0.8"
    assert np.corrcoef(pred, rate[train_end:])[0, 1] > 0.8

test_bins()
test_predict()
print("All checks passed")